In [42]:
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn import svm
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import RandomizedSearchCV

In [31]:
filename= 'datos_limpios.csv'
data= pd.read_csv(filename)
df=pd.DataFrame(data)
df.head()

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,...,room_type_Hotel room,room_type_Private room,room_type_Shared room,calendar_last_scraped_2025-09-16,calendar_last_scraped_2025-09-17,calendar_last_scraped_2025-09-22,calendar_last_scraped_2025-09-23,calendar_last_scraped_2025-09-24,calendar_last_scraped_2025-09-25,calendar_last_scraped_2025-09-30
0,1.0,2.0,30.26057,-97.73441,3,1.0,1.0,2.0,97.0,2,...,0,0,0,0,1,0,0,0,0,0
1,1.0,2.0,30.26034,-97.76487,2,1.0,1.0,2.0,160.0,3,...,0,0,0,0,1,0,0,0,0,0
2,1.0,1.0,30.23466,-97.73682,2,1.0,1.0,1.0,38.0,4,...,0,0,0,0,1,0,0,0,0,0
3,2.0,2.0,30.26098,-97.73072,3,2.0,2.0,2.0,145.0,15,...,0,0,0,0,1,0,0,0,0,0
4,1.0,1.0,30.23614,-97.73225,2,1.0,1.0,1.0,58.0,30,...,0,0,0,1,0,0,0,0,0,0


In [32]:
df = df.fillna(df.median(numeric_only=True))

In [33]:
#Se procede a la separación de las variables de esta manera,
#dado que según la instruccion del inciso 11, se ha de hacer la regresión directamente respecto al precio
y = df["price"]

#Variables predictoras
X = df.drop(columns=["price"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
columnas=X_train.columns
scaler = MinMaxScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [35]:
X_train=pd.DataFrame(X_train, columns=columnas)
X_test=pd.DataFrame(X_test, columns=columnas)

In [36]:
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1
inf = Q1 - 3 * IQR
sup = Q3 + 3 * IQR

PROBLEMA 10: tabla comparativa de los modelos

In [37]:
tabla=pd.DataFrame({
    "Modelo": ["Arbol de decisión","Random Forest","Naive Bayes","KNN","Regresión Logística"],
    "Predicciones correctas": [9099, 12524,7639,10358,12896],
    "Predicciones incorrectas":[4801,4318,7080,4361,1814 ]
})

In [38]:
tabla

,Modelo,Predicciones correctas,Predicciones incorrectas
0,Arbol de decisión,9099,4801
1,Random Forest,12524,4318
2,Naive Bayes,7639,7080
3,KNN,10358,4361
4,Regresión Logística,12896,1814


Es posible observar que la Regresión Logística es aquel modelo que tiene mayor número de aciertos y menor cantidad de errores. No obstante, como se ha mencionado en ese laboratorio, este modelo daba indicios de cierto sobreajuste. Le sigue el modelo de Random Forest, que en laboratorios anteriores se determinó que era el mejor modelo, ya que no estaba sobreajustado del todo y dada una cantidad de predicciones correctas lo suficientemente alta. Se observa que el modelo de KNN le seguiría a este ya que tiene una cantidad de aciertos elevada, pero la cantidad de predicciones incorrectas no es tan baja como para considerarlo sobreajuste. El modelo de árbol de decisión es cuarto en ser el mejor, ya que el número de predicciones incorrectas es casi la mitad en proporción de aquellas correctas. Finalmente se encuentra el modelo de Naive Bayes, el cual se puede observar que la cantidad de predicciones erradas es ligeramente mayor que la cantidad de predicciones correctas.

PROBLEMA 11: Modelo de Regresión

In [45]:
from sklearn.decomposition import PCA
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("svr", svm.SVR())
])

param_dist = {
    'svr__kernel': ['linear', 'rbf'],
    'svr__C': [0.5, 1, 10],
    'svr__gamma': ['scale', 0.5]
}

#Se usa Randomized Search para reducir el costo y el tiempo de cómputo
search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=2,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...svr', SVR())])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'svr__C': [0.5, 1, ...], 'svr__gamma': ['scale', 0.5], 'svr__kernel': ['linear', 'rbf']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 

In [29]:
print("Mejores parámetros:")
print(grid.best_params_)

Mejores parámetros:


AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== Métricas ===")
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)